In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit

In [0]:
path_energy  = "/mnt/silver/smart city/energy"
path_weather = "/mnt/silver/smart city/weather"
path_traffic = "/mnt/silver/smart city/traffic"

In [0]:
df_weather.printSchema()

root
 |-- reading_time: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- load_date: string (nullable = true)
 |-- api_status: string (nullable = true)
 |-- peak_hour_flag: integer (nullable = true)
 |-- weather_index: double (nullable = true)



In [0]:
df_final = (
    df_energy.alias("E")
    .join(df_weather.alias("W"), 
          col("E.reading_time") == col("W.reading_time"), 
          "left") 
    .join(df_traffic.alias("T"), 
          (col("E.reading_time") == col("T.timestamp")) & (col("E.region_id") == col("T.region_id")), 
          "left")
)

df_final_master = df_final.select(
    "E.reading_time",
    "E.region_id",
    "E.energy_consumption",
    "E.peak_hours_flag",
    "E.month",
    "E.weekday",
    
    col("W.temperature").alias("feels_like_raw"),
    col("W.humidity").alias("humidity_raw"),
    "W.weather_index",
    
    "T.congestion_index",
    "T.traffic_status"
)

print("FINAL TABLE IS READY!")
display(df_final_master)

df_final_master.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/mnt/silver/smart city/master_table")

FINAL TABLE IS READY!


reading_time,region_id,energy_consumption,peak_hours_flag,month,weekday,feels_like_raw,humidity_raw,weather_index,congestion_index,traffic_status
2025-12-01T19:00:00Z,r4,191.87507868761503,1,12,2,17.0,68.0,32.3,null,null
2025-12-03T11:00:00Z,r3,122.52461227547553,0,12,4,22.3,54.0,31.81,null,null
2025-12-03T13:00:00Z,r1,297.5346048500768,0,12,4,22.4,51.0,30.979999999999997,null,null
2025-12-03T18:00:00Z,r4,230.30063326928558,1,12,4,18.5,67.0,33.05,null,null
2025-12-05T19:00:00Z,r1,358.1714387939488,1,12,6,15.1,71.0,31.869999999999997,null,null
2025-12-05T21:00:00Z,r2,129.1763524431427,1,12,6,13.4,74.0,31.58,null,null
2025-12-02T00:00:00Z,r2,332.3918364745318,0,12,3,14.1,94.0,38.07,null,null
2025-12-03T10:00:00Z,r3,163.09665571073538,0,12,4,21.8,56.0,32.06,null,null
2025-12-03T00:00:00Z,r2,127.69553382794757,0,12,4,14.5,93.0,38.05,null,null
2025-12-05T15:00:00Z,r3,278.4211731808441,0,12,6,20.7,47.0,28.589999999999996,null,null


In [0]:
import pyspark.sql.functions as F

# 1. Calculate averages using the CORRECT column names (_raw)
avg_feels = df_final_master.agg(F.avg("feels_like_raw")).first()[0]
avg_humid = df_final_master.agg(F.avg("humidity_raw")).first()[0]
avg_congest = df_final_master.agg(F.avg("congestion_index")).first()[0]

# 2. Fill Nulls using the same (_raw) names
final_master_table = (
    df_final_master
    .fillna({
        "feels_like_raw": avg_feels if avg_feels else 0,
        "humidity_raw": avg_humid if avg_humid else 0,
        "congestion_index": avg_congest if avg_congest else 0,
        "traffic_status": "Unknown" 
    })
)

print("Nulls filled successfully.")
display(final_master_table)

Nulls filled successfully.


reading_time,region_id,energy_consumption,peak_hours_flag,month,weekday,feels_like_raw,humidity_raw,weather_index,congestion_index,traffic_status
2025-12-01T19:00:00Z,r4,191.87507868761503,1,12,2,17.0,68.0,32.3,0.0,Unknown
2025-12-03T11:00:00Z,r3,122.52461227547553,0,12,4,22.3,54.0,31.81,0.0,Unknown
2025-12-03T13:00:00Z,r1,297.5346048500768,0,12,4,22.4,51.0,30.979999999999997,0.0,Unknown
2025-12-03T18:00:00Z,r4,230.30063326928558,1,12,4,18.5,67.0,33.05,0.0,Unknown
2025-12-05T19:00:00Z,r1,358.1714387939488,1,12,6,15.1,71.0,31.869999999999997,0.0,Unknown
2025-12-05T21:00:00Z,r2,129.1763524431427,1,12,6,13.4,74.0,31.58,0.0,Unknown
2025-12-02T00:00:00Z,r2,332.3918364745318,0,12,3,14.1,94.0,38.07,0.0,Unknown
2025-12-03T10:00:00Z,r3,163.09665571073538,0,12,4,21.8,56.0,32.06,0.0,Unknown
2025-12-03T00:00:00Z,r2,127.69553382794757,0,12,4,14.5,93.0,38.05,0.0,Unknown
2025-12-05T15:00:00Z,r3,278.4211731808441,0,12,6,20.7,47.0,28.589999999999996,0.0,Unknown


In [0]:
df_final_master.write.format("delta").mode("overwrite").save("/mnt/silver/smart city/master_table")
print("Master Table Saved (Delta)")


Master Table Saved (Delta)


In [0]:
ml_path = "/mnt/silver/ml_ready_data"
df_final_master.write.format("parquet").mode("overwrite").save(ml_path)
print(f" ML Ready Data Saved at: {ml_path}")

 ML Ready Data Saved at: /mnt/silver/ml_ready_data


In [0]:
ml_parquet_path = "/mnt/silver/ml_ready_data"

df = spark.read.format("parquet").load(ml_parquet_path)

df.printSchema()

root
 |-- reading_time: timestamp (nullable = true)
 |-- region_id: string (nullable = true)
 |-- energy_consumption: double (nullable = true)
 |-- peak_hours_flag: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- weekday: integer (nullable = true)
 |-- feels_like_raw: double (nullable = true)
 |-- humidity_raw: double (nullable = true)
 |-- weather_index: double (nullable = true)
 |-- congestion_index: double (nullable = true)
 |-- traffic_status: string (nullable = true)



In [0]:
display(df)

reading_time,region_id,energy_consumption,peak_hours_flag,month,weekday,feels_like_raw,humidity_raw,weather_index,congestion_index,traffic_status
2025-12-01T19:00:00Z,r4,191.87507868761503,1,12,2,17.0,68.0,32.3,null,null
2025-12-03T11:00:00Z,r3,122.52461227547553,0,12,4,22.3,54.0,31.81,null,null
2025-12-03T13:00:00Z,r1,297.5346048500768,0,12,4,22.4,51.0,30.979999999999997,null,null
2025-12-03T18:00:00Z,r4,230.30063326928558,1,12,4,18.5,67.0,33.05,null,null
2025-12-05T19:00:00Z,r1,358.1714387939488,1,12,6,15.1,71.0,31.869999999999997,null,null
2025-12-05T21:00:00Z,r2,129.1763524431427,1,12,6,13.4,74.0,31.58,null,null
2025-12-02T00:00:00Z,r2,332.3918364745318,0,12,3,14.1,94.0,38.07,null,null
2025-12-03T10:00:00Z,r3,163.09665571073538,0,12,4,21.8,56.0,32.06,null,null
2025-12-03T00:00:00Z,r2,127.69553382794757,0,12,4,14.5,93.0,38.05,null,null
2025-12-05T15:00:00Z,r3,278.4211731808441,0,12,6,20.7,47.0,28.589999999999996,null,null


Silver Layer: final Master Table & ML Preparation

 Overview
This is the final step in the Data Engineering pipeline. It consolidates data from three disparate sources (Energy, Weather, Traffic) into a single "Source of Truth" table using timestamp alignment.

It also prepares the data for Machine Learning by applying Statistical Imputation to handle missing values, ensuring the ML model receives a complete dataset.

 Pipeline Steps
1. Data Integration (The Join Logic):

Primary Key: timestamp + region_id.

Join Type: Left Join on the Energy table.

Reason: To preserve all energy consumption records even if external factors (weather/traffic) are missing for a specific timestamp.

2. Handling Missing Values (Imputation): To avoid data loss (dropping rows) and ensure ML model stability, missing values were handled as follows:

Numerical Columns (feels_like, humidity, congestion): Filled with the Mean (Average) of the dataset.

Categorical Columns (traffic_status): Filled with "Unknown" to maintain schema consistency.

3. Output Deliverables:

Master Table (Delta): stored at /mnt/silver/smart city/master_table. Used for Power BI and Analytics.

ML Ready Data (Parquet): stored at /mnt/silver/ml_ready_data. Optimized format for Azure ML Designer ingestion.